<a href="https://colab.research.google.com/github/DrGPCR/NEUR201/blob/main/Unit_2/notebooks/Unit_2_Statistics_Assignment_STUDENT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unit 2 Statistics Assignment — Is the Drug Y Neuron Unusual?

**NEUR 201 — Research Methods & Data Analysis for Cellular Neuroscience**

**Name:** *(double-click to type)*  **Date:** *(double-click to type)*

---

## Background

In the Unit 2 Data Analysis Assignment you tested **Drug Y** by comparing one treated pain neuron with
one identical Control neuron. Real pain neurons are not identical: each receives a slightly different
amount of input and fires at its own rate. So before the lab can say that a Drug Y neuron fires
*differently*, they need to know how much **untreated** neurons vary on their own.

> **How likely is it that an untreated neuron would fire like the Drug Y neurons, just by chance?**

## The data

This is a **new recording session** — a different set of 200 untreated pain neurons from the one in
W6 L2, given a **stronger** stimulus — so expect different numbers throughout. Don't reuse your answers
from the lesson.

1. **200 untreated pain neurons.** Each neuron's input current varied from cell to cell, centred on about
   **0.46 nA** (`Input_current_nA`).
2. **A steady current ran for exactly 1 second**, and the spikes were counted, so the spike count **is**
   the firing rate in hertz (`Firing_rate_Hz`).

In Step 4 you'll compare these 200 untreated neurons with **200 Drug Y neurons** recorded under the same
conditions (`drug_y_firing_rates.csv`, same columns).

**Symbols:** $P$ = probability · $N_E$ = number of scores in the range · $N_T$ = total scores ·
$X$ = one raw score · $\bar{X}$ = sample mean · $s$ = sample standard deviation ·
$z$ = standard deviations from the mean

## How this notebook works

**Run every cell in order, from the top.** Click a cell and press **Shift + Enter**.

| | What to do |
|---|---|
| **Ordinary cells** | Just run them. You don't need to understand every line of code. |
| **💬 Discussion** | At the very end. Some questions ask you to **change a value in the code** and re-run, so find where it is set. |

**Points (10 total):** each of the 4 Steps is worth **1 pt** for running its cells (4 pts), and the
Discussion is worth **6 pts**.

If something breaks, first check that you ran **every cell above it**.

**To submit:** answer the Discussion, click **Runtime → Run all**, then **File → Print → Save as PDF**
and upload the PDF to Canvas.

---

## Step 1 — Load the data and look at it [1 pt]

This cell loads the dataset straight from the course GitHub page — nothing to download. `numpy` does the
maths, `pandas` holds the table, `matplotlib` draws the graphs, and `scipy.stats` gives us the
normal-distribution tools that replace the printed z-table.

Before computing anything clever, **look at the distribution**, and note its centre and spread. Almost
everything below is built from those two numbers.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

DATA = "https://raw.githubusercontent.com/DrGPCR/NEUR201/main/Unit_2/data/firing_rates_2.csv"

data = pd.read_csv(DATA)
rates = data["Firing_rate_Hz"].to_numpy(dtype=float)

mean = np.mean(rates)
sd = np.std(rates, ddof=1)      # ddof=1 because these 200 neurons are a SAMPLE

print("Loaded", len(data), "untreated pain neurons.\n")
print("Mean firing rate:    %.2f Hz" % mean)
print("Median firing rate:  %.2f Hz" % np.median(rates))
print("Standard deviation:  %.2f Hz" % sd)
print("Range:               %.0f to %.0f Hz" % (rates.min(), rates.max()))

BINS = 15

plt.figure(figsize=(7, 4))
plt.hist(rates, bins=BINS, color="lightsteelblue", edgecolor="black")
plt.axvline(mean, color="darkred", lw=2, label="mean")
plt.axvline(np.median(rates), color="navy", lw=2, ls="--", label="median")
plt.xlabel("Firing rate (Hz)")
plt.ylabel("Number of neurons")
plt.title("200 untreated pain neurons (%d bins)" % BINS)
plt.legend()
plt.show()

data.head()

## Step 2 — From frequency to probability [1 pt]

**Probability** is the likelihood of a particular event happening. In a sample of scores it is defined
as:

$$ P(\text{event}) = \frac{N_E}{N_T} $$

where $N_E$ is the number of times the outcome occurred and $N_T$ is the total number of scores.
Probability is just counting: if 4 of the 20 pain neurons you recorded fired above 80 Hz,
$P(\text{above 80 Hz}) = 4/20 = 0.2$.

This is the same calculation as the **relative frequency** ($rf = f/N$) from Unit 1. A relative frequency
table *is* a probability distribution — you were computing probabilities all along.

For a **continuous** variable like firing rate, the probability of one exact value isn't useful, so we
ask about a **range**:

$$ P(a < X < b) = \frac{N_E}{N_T} \qquad \text{where } N_E = \text{number of scores between } a \text{ and } b $$

In [ ]:
low, high = 75, 85

N_E = np.sum((rates >= low) & (rates <= high))   # neurons in the range
N_T = len(rates)                                 # total neurons
P = N_E / N_T

print("N_E (neurons firing between %d and %d Hz) = %d" % (low, high, N_E))
print("N_T (total neurons)                       = %d" % N_T)
print("P = N_E / N_T = %d / %d = %.3f" % (N_E, N_T, P))
print()
print("A randomly chosen untreated neuron has about a %.1f%% chance of firing in that range." % (100 * P))

plt.figure(figsize=(7, 4))
counts, bins, patches = plt.hist(rates, bins=15, density=True,
                                 color="lightsteelblue", edgecolor="black")
for patch, left in zip(patches, bins[:-1]):
    if low <= left < high:
        patch.set_facecolor("mediumseagreen")
plt.xlabel("Firing rate (Hz)")
plt.ylabel("Relative frequency (density)")
plt.title("Shaded bars = neurons between %d and %d Hz" % (low, high))
plt.show()

## Step 3 — Standardization: from a score to a probability [1 pt]

Firing rates like these form a bell-shaped **normal distribution**: symmetric, with the mean and median
at the peak. Because its shape is known exactly, we can turn any score into a probability. A **z-score**
expresses a raw score as a distance from the mean, in standard deviations:

$$ z = \frac{X - \bar{X}}{s} $$

A z-score of $+1.5$ means "one and a half standard deviations **above** the mean"; $-0.72$ means "0.72
standard deviations **below**." It changes the ruler, not the data.

Once a score is standardized, `norm.cdf(z)` gives the area under the standard normal curve below it —
the same number a printed z-table gives:

| Question | Code |
|---|---|
| Probability of a score **below** z | `norm.cdf(z)` |
| Probability of a score **above** z | `1 - norm.cdf(z)` |
| Probability **between** two z values | `norm.cdf(z2) - norm.cdf(z1)` |

In [ ]:
# Standardize every neuron in the dataset
z_scores = (rates - mean) / sd

plt.figure(figsize=(11, 3.8))
plt.subplot(1, 2, 1)
plt.hist(rates, bins=15, color="lightsteelblue", edgecolor="black")
plt.axvline(mean, color="darkred", lw=2)
plt.xlabel("Firing rate (Hz)")
plt.ylabel("Number of neurons")
plt.title("Raw scores")
plt.subplot(1, 2, 2)
plt.hist(z_scores, bins=15, color="mediumseagreen", edgecolor="black")
plt.axvline(0, color="darkred", lw=2)
plt.xlabel("z-score (SDs from the mean)")
plt.title("Standardized scores")
plt.tight_layout()
plt.show()


test_rate = 85

z = (test_rate - mean) / sd
p_below = norm.cdf(z)
p_above = 1 - p_below

print("A neuron firing at %g Hz:" % test_rate)
print("   z = (%g - %.2f) / %.2f = %+.2f" % (test_rate, mean, sd, z))
print("   It sits %.2f standard deviations %s the mean." % (abs(z), "above" if z > 0 else "below"))
print()
print("   P(firing slower) = %.4f  ->  about %.1f%% of untreated neurons fire slower" % (p_below, 100 * p_below))
print("   P(firing faster) = %.4f  ->  about %.1f%% of untreated neurons fire faster" % (p_above, 100 * p_above))

## Step 4 — Back to Drug Y: compare the two groups [1 pt]

Now the question this whole notebook has been building to. Here we have a **real Drug Y group**: 200 pain neurons, each exposed to Drug Y and given the typical stimulus for this session (about 0.46 nA) for 1 second, with the same cell-to-cell variation in input current.

Untreated neurons vary, so could their natural variation explain the Drug Y firing? We ask the **same probability question of both groups**, using $P = N_E / N_T$ from Step 2: *over the same range of firing rates, how likely is a randomly chosen neuron from each group to land in it?*

Then we put the Drug Y group **on the untreated ruler** with a z-score. The area under the curve beyond that z (a **tail area**) is a **p-value**: the probability, if only natural variation were at work, of a result at least this extreme.

In [ ]:
# Load the Drug Y group: same columns as the untreated file
DRUG_DATA = "https://raw.githubusercontent.com/DrGPCR/NEUR201/main/Unit_2/data/drug_y_firing_rates.csv"
drug = pd.read_csv(DRUG_DATA)
drug_rates = drug["Firing_rate_Hz"].to_numpy(dtype=float)

print("Loaded", len(drug), "Drug Y pain neurons.\n")
print("             untreated    Drug Y")
print("Mean (Hz)    %9.2f    %9.2f" % (mean, np.mean(drug_rates)))
print("Median (Hz)  %9.2f    %9.2f" % (np.median(rates), np.median(drug_rates)))
print("SD (Hz)      %9.2f    %9.2f" % (sd, np.std(drug_rates, ddof=1)))
print("Range (Hz)   %4.0f to %3.0f    %2.0f to %3.0f" % (rates.min(), rates.max(), drug_rates.min(), drug_rates.max()))
print()

# ---- The same probability question for both groups: P = N_E / N_T over the SAME range ----
RANGE_LOW, RANGE_HIGH = 75, 85

def probability_in_range(x, low, high):
    N_E = np.sum((x >= low) & (x <= high))     # neurons in the range
    N_T = len(x)                               # total neurons
    return N_E, N_T, N_E / N_T

for name, group in [("Untreated", rates), ("Drug Y", drug_rates)]:
    N_E, N_T, P = probability_in_range(group, RANGE_LOW, RANGE_HIGH)
    print("%-9s  P(%d-%d Hz) = %3d / %d = %.3f   (%.1f%% of neurons)" % (name, RANGE_LOW, RANGE_HIGH, N_E, N_T, P, 100 * P))
_, _, P_untreated = probability_in_range(rates, RANGE_LOW, RANGE_HIGH)
_, _, P_drug = probability_in_range(drug_rates, RANGE_LOW, RANGE_HIGH)

# ---- Put the Drug Y group on the untreated ruler: z-score, then the tail area (p-value) ----
z_drug_mean = (np.mean(drug_rates) - mean) / sd
p_tail = 1 - norm.cdf(z_drug_mean)       # area under the curve beyond z

print()
print("The Drug Y group's MEAN (%.2f Hz) sits at z = %+.2f (%.2f SDs %s the untreated mean)."
      % (np.mean(drug_rates), z_drug_mean, abs(z_drug_mean), "above" if z_drug_mean > 0 else "below"))
print("   Predicted share of UNTREATED neurons firing at least that fast: %.1f%%  (tail area, p = %.4f)"
      % (100 * p_tail, p_tail))
print("   Among our actual 200 untreated neurons: %d fired at least that fast."
      % np.sum(rates >= np.mean(drug_rates)))

# ---- Plot ----
SHARED_BINS = np.arange(60, 110, 5)

plt.figure(figsize=(7, 4))
plt.hist(rates, bins=SHARED_BINS, color="lightsteelblue", edgecolor="black", alpha=0.85,
         label="untreated")
plt.hist(drug_rates, bins=SHARED_BINS, color="crimson", edgecolor="black", alpha=0.55,
         label="Drug Y")
plt.axvspan(RANGE_LOW, RANGE_HIGH, color="mediumseagreen", alpha=0.25,
            label="range %d-%d Hz" % (RANGE_LOW, RANGE_HIGH))
plt.xlabel("Firing rate (Hz)")
plt.ylabel("Number of neurons")
plt.title("Untreated vs Drug Y, same bins")
plt.legend(fontsize=8)
plt.show()

### Your results

The numbers you need for the Discussion, in one table. **Re-run this cell** after you change any value above, so it shows your latest numbers.

In [ ]:
summary = pd.DataFrame([{
    "n": len(rates),
    "Mean_Hz": round(mean, 2),
    "Median_Hz": round(float(np.median(rates)), 2),
    "SD_Hz": round(sd, 2),
    "Test_rate_Hz": test_rate,
    "Test_rate_z": round(float(z), 2),
    "Test_rate_pct_predicted_faster": round(100 * p_above, 1),
    "Range_Hz": "%d-%d" % (RANGE_LOW, RANGE_HIGH),
    "P_untreated_in_range": round(float(P_untreated), 3),
    "P_drug_y_in_range": round(float(P_drug), 3),
    "Drug_Y_mean_Hz": round(float(np.mean(drug_rates)), 2),
    "Drug_Y_mean_z": round(float(z_drug_mean), 2),
    "Pct_untreated_predicted_this_fast": round(100 * float(p_tail), 1),
}]).astype(object)

summary.T          # .T flips the table on its side so it is easier to read

### 💬 Discussion [6 pts]

Keep each answer to **1–2 sentences** and use the numbers from your results. Questions 2, 3 and 4 ask for
a value that is **different from the one in the notebook**: find where it is set in the code, change it,
re-run that cell, then read the printed output (and re-run the results table).

1. **(Step 1)** Report the mean, median and SD of the untreated neurons. Based on the mean and median, is
   the distribution skewed? [1 pt]

2. **(Step 2)** What is the probability that a randomly chosen untreated neuron fires between **70 and
   80 Hz**? Give the $N_E$ and $N_T$ you divided. [1 pt]

3. **(Step 3)** For a neuron firing at **90 Hz**, report the z-score and the percentage of untreated
   neurons predicted to fire faster. [1 pt]

4. **(Step 4)** Set the range to **80 to 100 Hz**. Report the probability for the untreated neurons and
   for the Drug Y neurons, and the difference between them. Then answer **True or False**:

   > *"Drug Y has a lower probability than untreated neurons of firing between 80 and 100 Hz."* [2 pts]

5. **(Step 4)** Pain neurons signal pain by firing. Comparing the Drug Y neurons with the untreated
   neurons, is the pain-sensing neuron sensing **more** or **less** with Drug Y? Support your answer with
   the two group means. [1 pt]

**Your answer:** *(double-click here to type)*

>
>
>
>
>

---
### Nice work — you're done!

Click **Runtime → Run all**, then **File → Print → Save as PDF**, and upload the PDF to Canvas.

This cell is empty on purpose. Do not delete.